# Function info #

### PDFs ###

X'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

Z'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_social_bond_framework_series_ad_ae_2019.pdf'

V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/ISA_CTM_FRAMEWORK.pdf'

U'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/SONDA_S_A__GREEN_BOND_FRAMEWORK.pdf'

Portuguese

Y'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_Bonds_Framework_Athon_2020.pdf'

In [13]:
import pymupdf
import spacy
import re
import pandas as pd
import unicodedata


In [14]:
nlp = spacy.load('en_core_web_lg')

In [15]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances']

In [ ]:
# set keywords to search for start and end of UOP section

def set_kw_start(language):

    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
    elif language == 'PT':
        keywordsUOP = ['Uso de Recursos']
    elif language == 'ES':
        keywordsUOP = ['Uso de fondos', 'Uso de los fondos']
        
    return keywordsUOP
    
def set_kw_end(language):

    if language == 'EN':
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']
    elif language == 'PT':
        keywordsSEEGP = ['Processo de Avaliação e Seleção de Projetos'] # to be completed based on portuguese docs during refinement
    elif language == 'ES':
        keywordsSEEGP = ['XYZ'] # to be completed based on spanish docs during refinement

    return keywordsSEEGP



# find UOP start and end page number and rect coords based on keyword match

def find_uop_start(document, language):
    
    pdf = pymupdf.open(document)
    language = language

    areaUOP = None
    keywordsUOP = set_kw_start(language)

    if language == 'EN':                # functions for PT and ES to call the normalise, re-finditer and spac_to_doc
        for page_idx in range(len(pdf)):
            page = pdf[page_idx]

        if areaUOP is None:
            for keyword in keywordsUOP:
                start = page.search_for(keyword)
                if start:
                    areaUOP = (page_idx, start[0])
            return areaUOP

def find_uop_end(document, language):
    
    pdf = pymupdf.open(document)
    anguage = language

    areaSEEGP = None
    keywordsSEEGP = set_kw_end(language)

    if language == 'EN':
        for page_idx in range(len(pdf)):
            page = pdf[page_idx]

        if areaSEEGP is None:
            for keyword in keywordsSEEGP:
                end = page.search_for(keyword)
                if end:
                    areaSEEGP = (page_idx, end[0])

            return areaSEEGP



# iterate pages in the pdf and extract preferrably tables or else words from UOP section

def page_scenario_and_extract(document, language):

    pdf = pymupdf.open(document)
    language = language

    # inputs
    start_page_idx = find_uop_start(document, language)[0]
    end_page_idx = find_uop_end(document, language)[0]
    start_point = find_uop_start(document, language)[1].y1
    end_point = find_uop_end(document, language)[1].y0

    # outputs
    noTableMsg = []
    hasTableMsg = []
    hasDFMsg = []
    extractUOPwords = []

# iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

# process four page scenarios, check for tables, extract tables else extract text as words
    # A: UOP all on a single page
        if start_page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point and bbox[3] < end_point:
                    tableAheader = tables[0].header.names
                    tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
            else:
                noTableMsg.append('No tableA')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point and y1 < end_point:
                        extractUOPwords.append(text)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == start_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point:
                    tableBheader = tables[0].header.names
                    tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
            else:
                noTableMsg.append('No tableB')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point:
                        extractUOPwords.append(text)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > start_page_idx and page_idx < end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                tableDheader = tables[0].header.names
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
            else:
                noTableMsg.append(page_idx)
                noTableMsg.append('No tableD')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    extractUOPwords.append(text)

    # C: current page is end page
        elif page_idx == end_page_idx:
            tables = page.find_tables()
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[3] < end_point:
                    tableCheader = tables[0].header.names
                    tableCdf = tables[0].to_pandas()
                hasTableMsg.append(tableCheader)
                hasDFMsg.append(tableCdf)
            else:
                noTableMsg.append('No tableC')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y1 < end_point:
                        extractUOPwords.append(text)

    return noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords

    # noTableMsg lists page scenarios (and idx for scenario D) where no table is found
    # hasTableMsg lists the header row for found tables
    # hasDFMsg dataframe content in a list



# check extracted table is UOP table and extract Categories

def UOP_table_cats(hasDFMsg, hasTableMsg):

    # inputs
    hasDFMsg = hasDFMsg
    hasTableMsg = hasTableMsg

    # outputs
    uniqueCats = []
    errorMsg = []

    if len(hasDFMsg) != 1:
        errorMsg.append('More than one table found')

    # this over simplifies by assuming the category is always in the first column
    if 'Category' in hasTableMsg[0]:
        x = 'singular'
    elif 'Categories' in hasTableMsg[0]:
        x = 'plural'
    else:
        x = 'not a UOP table or category not in 1st columns'

    DFname = hasDFMsg[0]
    if x == 'singular':
        uniqueC = DFname['Category'].unique()
        uniqueCats.append(uniqueC)
    elif x == 'plural':
        uniqueC = DFname['Categories'].unique()
        uniqueCats.append(uniqueC)
    else:
        errorMsg.append(x)

    return errorMsg, uniqueCats

    # turn these into assert and proper error msgs later
        # if errorMsg is empty and uniqueCats contains a list of category like words, pdf has processed successfully
        # if errorMsg is not empty, there may be more than one table found, in which case DFname variable could be inaccurate
        # if errorMsg is not empty, the words Category or Categories may not be present in the header row, in which case may not be a UOP table or may be other words such as criteria



# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(extractUOPwords):

    # inputs
    extractUOPwords = extractUOPwords

    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}


    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim
                dfre = pd.DataFrame.from_dict(simsre, 'index')
                if dfre.empty:
                    reMsg = ['is not re']
                else:
                    reMsg = ['re word pairs']

    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
                dfee = pd.DataFrame.from_dict(simsee, 'index')
                if dfee.empty:
                    eeMsg = ['is not ee']
                else:
                    eeMsg = ['ee word pairs']
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim
                dfppc = pd.DataFrame.from_dict(simsppc, 'index')
                if dfppc.empty:
                    ppcMsg = ['is not ppc']
                else:
                    ppcMsg = ['ppc word pairs']

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim
                dfesml = pd.DataFrame.from_dict(simsesml, 'index')
                if dfesml.empty:
                    esmlMsg = ['is not esml']
                else:
                    esmlMsg = ['esml word pairs']

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim
                dftabc = pd.DataFrame.from_dict(simstabc, 'index')
                if dftabc.empty:
                    tabcMsg = ['is not tabc']
                else:
                    tabcMsg = ['tabc word pairs']

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim
                dfct = pd.DataFrame.from_dict(simsct, 'index')
                if dfct.empty:
                    ctMsg = ['is not ct']
                else:
                    ctMsg = ['ct word pairs']

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim
                dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')
                if dfswwm.empty:
                    swwmMsg = ['is not swwm']
                else:
                    swwmMsg = ['swwm word pairs']

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim
                dfcca = pd.DataFrame.from_dict(simscca, 'index')
                if dfcca.empty:
                    ccaMsg = ['is not cca']
                else:
                    ccaMsg = ['cca word pairs']

    # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim
                dfce = pd.DataFrame.from_dict(simsce, 'index')
                if dfce.empty:
                    ceMsg = ['is not ce']
                else:
                    ceMsg = ['ce word pairs']

    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim
                dfgb = pd.DataFrame.from_dict(simsgb, 'index')
                if dfgb.empty:
                    gbMsg = ['is not gb']
                else:
                    gbMsg = ['gb word pairs']

    return reMsg, dfre, eeMsg, dfee, ppcMsg, dfppc, esmlMsg, dfesml, tabcMsg, dftabc, ctMsg, dfct, swwmMsg, dfswwm, ccaMsg, dfcca, ceMsg, dfce, gbMsg, dfgb






In [28]:
# run the program

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'
language = 'EN'
result1 = find_uop_start(document, language)
result2 = find_uop_end(document, language)
result3 = set_kw_start(language)
result4 = set_kw_end(language)

print(result1)
print(result2)
print(result3)
print(result4)



(3, Rect(480.36907958984375, 187.17486572265625, 513.1909790039062, 199.73838806152344))
None
['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']
